# U.S. Electric Vehicle Market Share Analysis
**Role:** Data Analyst for a transportation research group  
**Goal:** Understand EV adoption trends across the U.S. and inform infrastructure planning  
**Dataset:** State-level vehicle registration counts by fuel type

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('Vehicle_Data.csv')

# Clean numeric columns (remove commas)
num_cols = df.columns[1:]
for col in num_cols:
    df[col] = df[col].astype(str).str.replace(',', '').str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

df.head()

## 1. Data Overview & Quality Check

In [ ]:
print(f"States/Territories: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nTotal vehicles registered: {df[num_cols].sum().sum():,}")

## 2. Market Share Analysis — EV vs. All Fuel Types

In [ ]:
# National totals by fuel type
totals = df[num_cols].sum().sort_values(ascending=False)

# EV share nationally
ev_share = (totals['Electric (EV)'] / totals.sum()) * 100
phev_share = (totals['Plug-In Hybrid Electric (PHEV)'] / totals.sum()) * 100
hev_share = (totals['Hybrid Electric (HEV)'] / totals.sum()) * 100
gas_share = (totals['Gasoline'] / totals.sum()) * 100

print("=== NATIONAL VEHICLE COMPOSITION ===")
print(f"Gasoline:        {gas_share:.2f}%")
print(f"EV:              {ev_share:.2f}%")
print(f"PHEV:            {phev_share:.2f}%")
print(f"HEV:             {hev_share:.2f}%")

# Pie chart
labels = ['Gasoline', 'EV', 'PHEV', 'HEV', 'Other']
sizes  = [
    totals['Gasoline'],
    totals['Electric (EV)'],
    totals['Plug-In Hybrid Electric (PHEV)'],
    totals['Hybrid Electric (HEV)'],
    totals[['Biodiesel','Ethanol/Flex (E85)','Compressed Natural Gas (CNG)',
             'Propane','Hydrogen','Methanol','Diesel','Unknown Fuel']].sum()
]

fig, ax = plt.subplots(figsize=(8, 8))
explode = (0, 0.05, 0, 0, 0)
ax.pie(sizes, labels=labels, autopct='%1.1f%%', explode=explode,
       colors=['#4a90d9','#2ecc71','#27ae60','#82e0aa','#bdc3c7'],
       startangle=140, textprops={'fontsize': 12})
ax.set_title('U.S. Vehicle Fleet Composition by Fuel Type', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('national_composition.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Top 10 States by EV Adoption Rate

In [ ]:
df['total_vehicles'] = df[num_cols].sum(axis=1)
df['ev_rate'] = (df['Electric (EV)'] / df['total_vehicles']) * 100

top10 = df.nlargest(10, 'ev_rate')[['State', 'Electric (EV)', 'total_vehicles', 'ev_rate']]
top10 = top10.reset_index(drop=True)
print(top10.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if i == 0 else '#4a90d9' for i in range(10)]
bars = ax.barh(top10['State'][::-1], top10['ev_rate'][::-1], color=colors[::-1])
ax.set_xlabel('EV Adoption Rate (%)', fontsize=12)
ax.set_title('Top 10 States by EV Adoption Rate', fontsize=14, fontweight='bold')
for bar, val in zip(bars, top10['ev_rate'][::-1]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('top10_ev_states.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. California vs. Large States Comparison

In [ ]:
compare_states = ['California', 'Texas', 'Florida', 'New York', 'Washington']
compare = df[df['State'].isin(compare_states)][['State', 'Electric (EV)', 'total_vehicles', 'ev_rate']].copy()
compare = compare.sort_values('ev_rate', ascending=False)
print(compare.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# EV count
axes[0].bar(compare['State'], compare['Electric (EV)'] / 1e6,
            color='#4a90d9', edgecolor='white')
axes[0].set_title('EV Registrations (Millions)', fontweight='bold')
axes[0].set_ylabel('Vehicles (M)')

# EV rate
axes[1].bar(compare['State'], compare['ev_rate'],
            color='#2ecc71', edgecolor='white')
axes[1].set_title('EV Adoption Rate (%)', fontweight='bold')
axes[1].set_ylabel('Adoption Rate (%)')

for ax in axes:
    ax.tick_params(axis='x', rotation=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('EV Landscape: California vs. Major States', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('state_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Alternative Fuels: Significant vs. Niche

In [ ]:
alt_fuels = ['Biodiesel','Ethanol/Flex (E85)','Compressed Natural Gas (CNG)',
             'Propane','Hydrogen','Methanol']
alt_totals = df[alt_fuels].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(alt_totals.index, alt_totals.values / 1e3,
       color=['#e67e22','#f39c12','#95a5a6','#bdc3c7','#d5d8dc','#ecf0f1'],
       edgecolor='white')
ax.set_ylabel('Registrations (Thousands)')
ax.set_title('Alternative Fuel Vehicles: National Totals', fontweight='bold')
ax.tick_params(axis='x', rotation=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

threshold = alt_totals.max() * 0.1
for i, (fuel, val) in enumerate(alt_totals.items()):
    label = 'SIGNIFICANT' if val > threshold else 'niche'
    ax.text(i, val/1e3 + alt_totals.max()/1e3 * 0.01, label,
            ha='center', fontsize=8, color='#2c3e50')

plt.tight_layout()
plt.savefig('alt_fuels.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Infrastructure Investment Recommendations

In [ ]:
# High EV adoption + Low PHEV support = needs charging infra urgently
df['alt_support'] = df[['Biodiesel','Compressed Natural Gas (CNG)','Propane','Hydrogen']].sum(axis=1)
df['alt_support_rate'] = df['alt_support'] / df['total_vehicles'] * 100

# Recommendation: top EV states where alternative support is still low
reco = df[df['ev_rate'] > df['ev_rate'].quantile(0.6)].copy()
reco = reco.sort_values('ev_rate', ascending=False)[['State','ev_rate','alt_support_rate']].head(10)

print("=== TOP 3 INFRASTRUCTURE INVESTMENT PRIORITIES ===\n")
top3 = reco.head(3)
for _, row in top3.iterrows():
    print(f"  {row['State']}")
    print(f"    EV Adoption Rate:         {row['ev_rate']:.2f}%")
    print(f"    Alt. Support Coverage:    {row['alt_support_rate']:.3f}%")
    print(f"    Rationale: High EV demand but minimal alternative fuel infrastructure.\n")

## 7. Key Insights & Conclusion

- **EVs are still a small share** of the total U.S. fleet but growing, with clear leaders like California and Washington.
- **California dominates** in absolute EV count but some smaller states show higher adoption *rates*.
- **Ethanol/Flex (E85)** is the most significant alternative fuel by volume; all others are niche.
- **Infrastructure gap:** The states with the highest EV adoption often have minimal complementary alternative fuel support, making them prime targets for charging network investment.
- These findings directly inform where policymakers should prioritise EV charging rollouts — relevant to transmission and energy infrastructure planning.
